**Redrob AI INDIA.RUNS**

Offline Candidate Index Generation
This notebook generates semantic retrieval artifacts for the complete candidate database.
Outputs
- embeddings
- faiss index
- candidate lookup
- metadata

In [1]:
from pathlib import Path
import os
import sys

In [2]:
CODEBASE = Path(
    "/kaggle/input/datasets/parmindersingh2002/redrob-ai-india-runs-hackathon-codebase"
)

DATASET = Path(
    "/kaggle/input/datasets/parmindersingh2002/redrob-ai-india-runs-hackathon-dataset"
)

print(CODEBASE.exists())
print(DATASET.exists())

True
True


In [3]:
import shutil

WORKDIR = Path("/kaggle/working/INDIA-RUNS")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

shutil.copytree(CODEBASE, WORKDIR)

print(WORKDIR)

/kaggle/working/INDIA-RUNS


In [4]:
%cd /kaggle/working/INDIA-RUNS

!pip install -q --upgrade pip

!pip install -q -r requirements.txt

/kaggle/working/INDIA-RUNS
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.7 MB/s eta 0:00:00


In [5]:
import sys

sys.path.append(str(WORKDIR))

print(sys.path[-1])

/kaggle/working/INDIA-RUNS


In [6]:
from src.config import (
    PROJECT_ROOT,
    ARTIFACTS_DIR,
    EMBEDDINGS_DIR,
    FAISS_DIR,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("EMBEDDINGS_DIR:", EMBEDDINGS_DIR)
print("FAISS_DIR:", FAISS_DIR)

ARTIFACTS_DIR.mkdir(exist_ok=True)
EMBEDDINGS_DIR.mkdir(exist_ok=True)
FAISS_DIR.mkdir(exist_ok=True)

print("Directories ready.")

PROJECT_ROOT: /kaggle/working/INDIA-RUNS
ARTIFACTS_DIR: /kaggle/working/INDIA-RUNS/artifacts
EMBEDDINGS_DIR: /kaggle/working/INDIA-RUNS/artifacts/embeddings
FAISS_DIR: /kaggle/working/INDIA-RUNS/artifacts/faiss
Directories ready.


In [7]:
from src.pipeline.offline_pipeline import OfflineIndexBuilder
from src.retrieval.retriever import Retriever

In [8]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [9]:
CANDIDATES = DATASET / "candidates.jsonl"

print(CANDIDATES)

print(CANDIDATES.exists())

/kaggle/input/datasets/parmindersingh2002/redrob-ai-india-runs-hackathon-dataset/candidates.jsonl
True


In [10]:
builder = OfflineIndexBuilder()

result = builder.build_candidate_index(
    candidates_jsonl_path=CANDIDATES,
    max_candidates=None,
    force_rebuild=True,
)

Loading candidates...
Parsing 100000 candidates...
Building retrieval documents...
Generating embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Building FAISS index...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Building lookup: 100%|██████████| 100000/100000 [00:00<00:00, 1075346.76it/s]


Saving artifacts...
✓ All validation checks passed
Completed.


In [11]:
print(result)

OfflineIndexResult
Total Candidates: 100000
Processed Candidates: 100000
Embedding Dimension: 768
Index Size: 100000
Processing Time: 4379.09s
  - Embedding Time: 2166.14s
  - Index Build Time: 2167.98s

Artifacts:
  faiss_index: /kaggle/working/INDIA-RUNS/artifacts/faiss/faiss.index
  candidate_lookup: /kaggle/working/INDIA-RUNS/artifacts/faiss/candidate_lookup.pkl
  embedding_metadata: /kaggle/working/INDIA-RUNS/artifacts/faiss/embedding_metadata.pkl

Statistics:
  parse_time_seconds: 9.564294815063477
  document_generation_time_seconds: 25.275079488754272
  average_embedding_time_seconds: 0.021661415996551515


In [12]:
print("Processed:", result.processed_candidates)

print("Embedding dimension:", result.embedding_dimension)

print("Index size:", result.index_size)

print()

print("Artifacts")

for k, v in result.artifacts.items():
    print(k, "->", v)

Processed: 100000
Embedding dimension: 768
Index size: 100000

Artifacts
faiss_index -> /kaggle/working/INDIA-RUNS/artifacts/faiss/faiss.index
candidate_lookup -> /kaggle/working/INDIA-RUNS/artifacts/faiss/candidate_lookup.pkl
embedding_metadata -> /kaggle/working/INDIA-RUNS/artifacts/faiss/embedding_metadata.pkl


In [13]:
retriever = Retriever()

retriever.load_index()

print("Index loaded.")

Index loaded.


In [14]:
results = retriever.search(

    "Machine Learning Engineer Python Deep Learning",

    k=5

)

for r in results:

    print(r)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'rank': 1, 'candidate_id': 'CAND_0013613', 'similarity': 0.756229817867279, 'metadata': {'candidate_id': 'CAND_0013613', 'title': 'Machine Learning Engineer', 'location': 'Singapore', 'experience_years': 4.7, 'number_of_skills': 14, 'number_of_career_entries': 3, 'number_of_education_entries': 1, 'document_length': 2143, 'document_word_count': 283, 'section_count': 8, 'sections': ['title', 'summary', 'current_role', 'career_history', 'skills', 'industries', 'experience', 'education'], 'current_title': 'Machine Learning Engineer', 'current_company': 'Adobe', 'current_industry': 'Software'}}
{'rank': 2, 'candidate_id': 'CAND_0089552', 'similarity': 0.7477204203605652, 'metadata': {'candidate_id': 'CAND_0089552', 'title': 'Machine Learning Engineer', 'location': 'Kolkata, West Bengal', 'experience_years': 6.0, 'number_of_skills': 19, 'number_of_career_entries': 2, 'number_of_education_entries': 1, 'document_length': 1668, 'document_word_count': 227, 'section_count': 8, 'sections': ['titl

/kaggle/working/INDIA-RUNS/src/embeddings/embedder.py:98: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._embedding_dim = self._model.get_sentence_embedding_dimension()


In [15]:
import time

start = time.time()

results = retriever.search(
    "Machine Learning Engineer Python Deep Learning",
    k=10,
)

elapsed = time.time() - start

print(f"Search latency: {elapsed:.4f} sec")

for r in results[:5]:
    print(r)

Search latency: 0.0499 sec
{'rank': 1, 'candidate_id': 'CAND_0013613', 'similarity': 0.756229817867279, 'metadata': {'candidate_id': 'CAND_0013613', 'title': 'Machine Learning Engineer', 'location': 'Singapore', 'experience_years': 4.7, 'number_of_skills': 14, 'number_of_career_entries': 3, 'number_of_education_entries': 1, 'document_length': 2143, 'document_word_count': 283, 'section_count': 8, 'sections': ['title', 'summary', 'current_role', 'career_history', 'skills', 'industries', 'experience', 'education'], 'current_title': 'Machine Learning Engineer', 'current_company': 'Adobe', 'current_industry': 'Software'}}
{'rank': 2, 'candidate_id': 'CAND_0089552', 'similarity': 0.7477204203605652, 'metadata': {'candidate_id': 'CAND_0089552', 'title': 'Machine Learning Engineer', 'location': 'Kolkata, West Bengal', 'experience_years': 6.0, 'number_of_skills': 19, 'number_of_career_entries': 2, 'number_of_education_entries': 1, 'document_length': 1668, 'document_word_count': 227, 'section_co

In [16]:
print("Total")

print(result.processing_time_seconds)

print()

print("Embedding")

print(result.embedding_time_seconds)

print()

print("Index")

print(result.index_build_time_seconds)

Total
4379.086310386658

Embedding
2166.1415996551514

Index
2167.980366706848


In [17]:
!find artifacts

artifacts
artifacts/faiss
artifacts/faiss/candidate_lookup.pkl
artifacts/faiss/embedding_metadata.pkl
artifacts/faiss/faiss.index
artifacts/embeddings
artifacts/embeddings/test_embeddings.npz


In [18]:
import shutil

shutil.make_archive(

    "/kaggle/working/redrob_artifacts",

    "zip",

    "artifacts"

)

print("Created redrob_artifacts.zip")

Created redrob_artifacts.zip
